# Set 1 오답노트

## 검토한 풀이 파일

- `00_trying/01/01_question.ipynb`
- `00_trying/02/01_question.ipynb`
- `00_trying/02/01_question copy.ipynb`
- `set_01_06_answer/01_question.ipynb`

## 최종 답

- Q01: **510개**
- Q02: **0.38**
- Q03: **할인율(`current_price / MRP`)**


## Q01 — 주사율 문자열 추출

### 문제에서 보장한 조건만 사용하기

주사율은 2~3자리 숫자 뒤에 `Hz`가 붙는다고 했으므로 `Refresh Rate`까지 항상 있다고 가정하지 않는다. 세 열을 합칠 때는 값이 붙지 않도록 열 사이에 공백을 넣는다.

```python
df_q1 = df.copy()
df_q1['f-p-s'] = (
    df_q1['Frequency'] + ' ' +
    df_q1['Picture_quality'] + ' ' +
    df_q1['Speaker']
)
```

### `isin`, `contains`, `extract` 구분

- `isin()`: 셀의 값 전체가 목록에 있는 값과 일치하는지 검사
- `str.contains()`: 긴 문자열 안에 패턴이 포함되는지 검사
- `str.extract()`: 패턴 중 캡처 그룹 `()` 안의 값을 추출

숫자만 추출하는 권장 풀이:

```python
freq = df_q1['f-p-s'].str.extract(
    r'(?P<Frequency>\d{2,3})\s*Hz',
    expand=False
)
answer_q1 = freq.eq('60').sum()
display(answer_q1)  # 510
```

- `\d{2,3}`: 숫자 2~3자리
- `\s*`: 공백 0개 이상
- `Hz`: 추출값이 아니라 패턴 확인 조건
- `(?P<Frequency>...)`: 캡처 그룹의 이름 지정
- `expand=False`: 결과를 1차원 Series로 반환

`Hz`까지 추출하려면 `r'(?P<Frequency>\d{2,3}\s*Hz)'`처럼 `Hz`도 괄호 안에 넣는다.

### DataFrame으로 추출했을 때 열 이름 지정

`expand`의 기본값은 `True`이므로 결과가 DataFrame이고, 이름 없는 그룹의 열 이름은 `0`이다.

```python
freq_df = df_q1['f-p-s'].str.extract(r'(\d{2,3})\s*Hz')
freq_df = freq_df.rename(columns={0: 'freq'})
answer_q1 = freq_df['freq'].eq('60').sum()
```

`freq_df.isin(['60']).sum()`은 열별 합계인 Series를 반환한다. 최종 숫자 하나가 필요하면 열을 선택한 뒤 비교한다. 추출 결과는 문자열이므로 `60`이 아닌 `'60'`과 비교한다.

### `contains()`만 사용하는 풀이

```python
answer_q1 = df_q1['f-p-s'].str.contains(
    r'(?<!\d)60\s*Hz\b',
    na=False
).sum()
```

`'60 Hz'`만 단순 검색하면 공백 없는 `60Hz`를 놓칠 수 있다. 숫자 경계를 검사하지 않으면 `160 Hz` 같은 값까지 잘못 포함할 수 있다.


## Q02 — 8K와 4K 평균 평점 차이

해상도 정보가 세 열 중 어디에 들어 있어도 검색할 수 있도록 공백을 넣어 합친다. 각 조건에 해당하는 `Stars`의 평균을 구한 뒤 절댓값과 반올림을 적용한다.

```python
df_q2 = df.copy()
df_q2['o-c-p'] = (
    df_q2['Operating_system'] + ' ' +
    df_q2['channel'] + ' ' +
    df_q2['Picture_quality']
)

cond_8k = df_q2['o-c-p'].str.contains('8K', na=False)
cond_4k = df_q2['o-c-p'].str.contains('4K', na=False)

stars_8k = df_q2.loc[cond_8k, 'Stars'].mean()
stars_4k = df_q2.loc[cond_4k, 'Stars'].mean()
answer_q2 = round(abs(stars_8k - stars_4k), 2)
display(answer_q2)  # 0.38
```

차이의 크기를 묻기 때문에 `abs()`를 사용하고, 계산 중간이 아닌 최종 결과에서 `round(..., 2)`를 적용한다.


## Q03 — 조건 결합, 결측치, Random Forest

### OR, AND, NOT 조건

```python
# OR
cond_or = df['channel'].str.contains(r'Pixel|Oper', na=False)

# AND
cond_and = (
    df['channel'].str.contains('Netflix', na=False)
    & df['channel'].str.contains('Youtube', na=False)
)

# NOT
df_filtered = df.loc[~cond_or].copy()
```

Series 조건은 `or`, `and`, `not`이 아니라 `|`, `&`, `~`를 사용한다. 조건을 따로 작성할 때는 각각 괄호로 감싼다. 정규식의 `4K|8K`에서 `|`도 OR를 뜻한다.

### 필터링 후 파생변수 만들기

```python
df_q3 = df.loc[~df['channel'].str.contains(r'Pixel|Oper', na=False)].copy()

df_q3['review_ratio'] = df_q3['Reviews'] / df_q3['Ratings']
df_q3['discount_rate'] = df_q3['current_price'] / df_q3['MRP']
df_q3['Netflix'] = df_q3['channel'].str.contains('Netflix', na=False).astype(int)
df_q3['PrimeVideo'] = df_q3['channel'].str.contains('Prime Video', na=False).astype(int)
df_q3['high_quality'] = df_q3['Picture_quality'].str.contains(
    r'4K|8K', na=False
).astype(int)
```

필터링된 DataFrame에서 직접 파생변수를 만들면 인덱스 정렬에 의존하지 않아 코드의 흐름이 분명하다. `str.contains()`의 Boolean 결과에 `.astype(int)`를 적용하면 `True → 1`, `False → 0`이 된다.

### 결측치 확인과 제거

- `isna().sum()`: 결측치 개수
- `isna().any()`: 하나라도 결측치가 있는지
- `isna().all()`: 모든 값이 결측치인지
- `axis=1`: 열별이 아닌 행별로 검사

```python
cols_X = [
    'review_ratio', 'MRP', 'discount_rate',
    'Netflix', 'PrimeVideo', 'high_quality'
]
model_cols = cols_X + ['Stars']
df_model = df_q3.dropna(subset=model_cols).copy()
display(df_model.shape)  # 행 개수 197
```

`dropna()`를 전체 열에 적용하면 모델에서 사용하지 않는 열의 결측치 때문에 행이 제거될 수 있다. 모델에 사용할 열만 `subset`으로 지정하는 편이 안전하다.

### 단일 종속변수 `y`는 1차원으로 만들기

```python
X = df_model[cols_X]       # shape: (197, 6)
y = df_model['Stars']      # shape: (197,)
```

`df_model[['Stars']]`는 `(197, 1)`인 2차원 DataFrame이므로 Random Forest에서 `A column-vector y was passed when a 1d array was expected` 경고가 발생한다. `df_model['Stars']`처럼 Series로 전달한다. 이미 2차원으로 만든 경우에는 `y.squeeze()` 또는 `y.values.ravel()`을 사용할 수 있다.

### 변수 중요도 확인

```python
model = RandomForestRegressor(random_state=123)
model.fit(X, y)

importance = pd.Series(
    model.feature_importances_,
    index=cols_X
).sort_values(ascending=False)

display(importance)
display(importance.idxmax())  # discount_rate
```

Series 정렬은 `sort_values()`를 사용한다. 최댓값에 대응하는 변수명은 `idxmax()`, 중요도 값은 `max()`로 구한다. 이 문제는 예측값을 요구하지 않으므로 `predict()`는 필요 없다.


## 핵심 암기

1. 값 전체 비교는 `isin()`, 포함 여부는 `str.contains()`, 일부 추출은 `str.extract()`를 사용한다.
2. `extract()`는 괄호 안만 반환하고 `expand=False`이면 Series가 된다.
3. 문자열 검색 조건의 OR·AND·NOT은 `|`, `&`, `~`로 결합한다.
4. Boolean을 이진 변수로 바꿀 때 `.astype(int)`를 사용한다.
5. 모델용 결측치 제거는 `dropna(subset=모델에_사용할_열)`로 범위를 명시한다.
6. 단일 종속변수 `y`는 `df['column']` 형태의 1차원 Series로 전달한다.
7. 변수 중요도는 `pd.Series(model.feature_importances_, index=cols_X)`로 확인한다.
